# Sales Lead Assistant (RAG + Memory)
### Mid-term Project - Hagar Soliman

**Idea:** a chatbot inside the CRM of *Nova Digital Solutions*. Before a call, the sales rep asks for a lead brief or advice.

| Source | What it has |
|---|---|
| Google Sheet | 200 leads (company, budget, stage, notes ...) |
| PDF Playbook | services, prices, discounts, sales rules |
| Memory | remembers the last 5 messages (for follow-up questions) |

**Flow:** Question -> find the lead(s) -> search the playbook -> build prompt (+ memory) -> Qwen answers -> check the prices -> save in memory

**What is new in this version**
- No deprecated libraries (`langchain-community` is being sunset): FAISS and sentence-transformers are used directly.
- The data files download by themselves, so the notebook runs with **Runtime -> Run all**.
- Counting questions ("how many leads are in Negotiation?") are answered with **pandas**, not guessed by the model.
- Two companies in one question ("compare Nile Pharma and Sinai Tours") work.
- Company names are found even with different case, accents or a small typo.
- Every playbook chunk knows its **page**, and every answer can show its **sources**.
- A **price check** catches a wrong price for a package and asks the model again.
- An **at-risk** flag (no contact for more than 14 days) from the playbook.
- The evaluation has **new questions that were not used to write the prompt**, to measure accuracy honestly.

## 1. Setup
Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
!pip install -q langchain-text-splitters faiss-cpu sentence-transformers pypdf bitsandbytes accelerate   # install the libraries we need (-q = quiet)

In [ ]:
import os, re, time, difflib, unicodedata, urllib.request                 # standard library: files, regex, timing, fuzzy matching, accents, downloads
from datetime import date                                                 # dates (for "at risk")
import numpy as np                                                        # numbers / vectors
import pandas as pd                                                       # pandas: work with the leads table
import torch                                                              # PyTorch: runs the model on the GPU
import faiss                                                              # FAISS: vector search by meaning
from pypdf import PdfReader                                               # reads the PDF playbook
from sentence_transformers import SentenceTransformer                     # turns text into vectors (embeddings)
from langchain_text_splitters import RecursiveCharacterTextSplitter       # cuts long text into small chunks
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig   # the LLM (Qwen) + its tokenizer + 4-bit settings

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"   # stop early with a clear message
print("GPU:", torch.cuda.get_device_name(0))                              # show which GPU we got

## 2. Load and Clean the Data
The files are downloaded from the project's GitHub repo if they are not already in `/content/`.
The leads are read from the Google Sheet; if the sheet is not reachable, the Excel copy is used, so the notebook never stops here.

In [ ]:
REPO = "https://raw.githubusercontent.com/Hagar202/nova-sales-assistant/main/data/"   # copy of the data on GitHub
pdf_path = "/content/Nova_Digital_Sales_Playbook.pdf"                    # the playbook PDF
xlsx_path = "/content/Nova_Digital_Leads.xlsx"                           # Excel copy of the leads sheet
sheet_url = "https://docs.google.com/spreadsheets/d/1PzyqWRR20jeSbgWrxTDG2x96rY7YJKbVjGXctUwirw4/export?format=csv"   # Google Sheet exported as CSV

for path in [pdf_path, xlsx_path]:                                       # download each file once
    if not os.path.exists(path):
        urllib.request.urlretrieve(REPO + os.path.basename(path), path)

try:
    df = pd.read_csv(sheet_url)                                          # live data from the Google Sheet
    print("Leads read from the Google Sheet")
except Exception as error:                                               # no internet / sheet not shared -> use the Excel copy
    df = pd.read_excel(xlsx_path)
    print("Google Sheet not reachable (", type(error).__name__, ") -> using the Excel copy")

pdf_pages = [(p.extract_text() or "").replace("\x7f", "-") for p in PdfReader(pdf_path).pages]   # one text per page; PDF bullets -> "-"
print("PDF pages:", len(pdf_pages), "| Leads:", len(df))                 # quick check: how many pages and how many leads

In [ ]:
df.columns = df.columns.str.strip()                                          # remove extra spaces from column names
df = df.apply(lambda c: c.str.strip() if c.dtype == "object" else c)         # remove extra spaces from every text cell
df = df.drop_duplicates(subset="Lead ID").reset_index(drop=True)             # remove duplicate leads (same Lead ID) and renumber rows
df["Budget (USD)"] = pd.to_numeric(df["Budget (USD)"], errors="coerce")      # make budget a number (bad values -> NaN)
df["Employees"] = pd.to_numeric(df["Employees"], errors="coerce")            # make employees a number
df["Last Contact"] = pd.to_datetime(df["Last Contact"], errors="coerce").dt.strftime("%Y-%m-%d")   # one date format YYYY-MM-DD
print(len(df), "leads |", int(df.isna().sum().sum()), "missing values")     # check: number of leads + number of empty cells

## 3. Rules From the Playbook (computed in code, not by the LLM)
Budget check, decision maker, priority, next action and **at risk** are plain Python.
The model gets them as `COMPUTED FACTS` and is told to trust them, so it never has to calculate.

In [ ]:
TODAY = date(2026, 10, 22)        # the CRM sheet is a snapshot from this day; "at risk" is counted from it (use date.today() for live data)

# price of every package (from the playbook)
PRICES = {"Basic Website":1500,"Business Website":4000,"E-commerce Store":8000,               # website packages
          "Basic App":6000,"Standard App":12000,"Advanced App":25000,                         # app packages
          "Social Media Management":800,"Ads Management":1200,"SEO":1000,                     # marketing packages (per month)
          "AI Chatbot":5000,"Business Automation":7000,"CRM + AI Integration":10000}          # AI packages
MONTHLY = {"Social Media Management", "Ads Management", "SEO"}                                # packages priced per month
# next step for the rep based on the lead stage (playbook section 8)
NEXT_ACTION = {"New":"Call the lead within 24 hours",                                         # new lead -> call fast
               "Contacted":"Book a discovery meeting within 3 days",                          # contacted -> book a meeting
               "Proposal Sent":"Follow up within 3 days if there is no reply",                # proposal sent -> follow up
               "Negotiation":"Follow the discount rules and close the deal",                  # negotiation -> close
               "Won":"Hand over to the project manager",                                      # won -> hand over
               "Lost":"Follow up again after 90 days"}                                        # lost -> try again later
# response time for NEW leads depends on the source (playbook section 14)
NEW_BY_SOURCE = {"Referral":"Call the lead within 4 hours (referrals are the best leads)",
                 "Exhibition":"Contact the lead within 48 hours after the exhibition",
                 "Cold Call":"Call within 24 hours and start by explaining the problem we solve"}
# packages grouped by service family (used to suggest a cheaper package of the SAME service)
FAMILY = {"Website": ["Basic Website","Business Website","E-commerce Store"],                 # websites
          "App": ["Basic App","Standard App","Advanced App"],                                 # apps
          "AI": ["AI Chatbot","Business Automation","CRM + AI Integration"],                  # AI
          "Marketing": ["SEO","Social Media Management","Ads Management"]}                    # marketing
DM_WORDS = ["ceo","owner","founder","director","manager"]      # job titles that count as decision makers (playbook section 7)
AT_RISK_DAYS = 14                                              # playbook section 8: no contact for more than 14 days = at risk

def price_text(service):                                       # "$25,000", "$1,000 / month", "$5,000 + $300 / month"
    if service == "AI Chatbot": return "$5,000 + $300 / month"
    return f"${PRICES[service]:,}" + (" / month" if service in MONTHLY else "")

def cheaper_options(service):                                  # cheaper packages in the same family, closest price first
    for pk in FAMILY.values():
        if service in pk:
            low = sorted([s for s in pk if PRICES[s] < PRICES[service]], key=lambda s: -PRICES[s])
            return ", ".join(f"{s} ({price_text(s)})" for s in low) or "none"
    return "none"

def compute(i):                                                # all the rules for lead number i, as a dict
    r = df.iloc[i]                                                               # the lead row
    price = PRICES.get(r["Interested Service"])                                  # price of the service they want
    budget_ok = price is not None and r["Budget (USD)"] >= price                 # budget >= price ?
    dm = any(w in str(r["Job Title"]).lower() for w in DM_WORDS)                 # is the contact a decision maker?
    qualified = budget_ok and dm                                                 # qualified = budget OK + decision maker
    stage = r["Stage"]
    if stage in ("Won", "Lost"):  priority = stage                              # deal finished
    elif not qualified:           priority = "Cold"                             # not qualified -> cold
    elif stage in ("Proposal Sent", "Negotiation"): priority = "Hot"            # qualified + close to closing -> hot
    else:                         priority = "Warm"                             # qualified but early stage -> warm
    days = max(0, (TODAY - date.fromisoformat(r["Last Contact"])).days)          # days since the last contact
    at_risk = stage not in ("Won", "Lost") and days > AT_RISK_DAYS               # active lead forgotten for > 14 days
    action = NEW_BY_SOURCE.get(r["Lead Source"], NEXT_ACTION[stage]) if stage == "New" else NEXT_ACTION[stage]
    if not dm:   action = "Reach a decision maker first (contact is not a decision maker)"   # wrong person -> find the decision maker
    elif not budget_ok: action += "; budget is below the price, suggest a cheaper package"   # low budget -> cheaper package
    if at_risk: action = f"AT RISK ({days} days without contact): contact them today. " + action
    return dict(price=price, budget_ok=budget_ok, dm=dm, qualified=qualified, priority=priority,
                days=days, at_risk=at_risk, action=action)

facts = pd.DataFrame([compute(i) for i in range(len(df))])     # the rules for every lead, in one table
df["Priority"], df["At Risk"] = facts["priority"], facts["at_risk"]   # add them to the leads table (used for counting questions)

def lead_facts(i):                                             # one line of computed facts for the prompt
    r, f = df.iloc[i], facts.iloc[i]
    cross = ("Social Media Management or Ads Management"                         # cross-sell for website/app clients: marketing
             if any(w in r["Interested Service"] for w in ("Website", "App", "E-commerce")) else "a related service from the playbook")
    return (f"COMPUTED FACTS for {r['Company']} (trust these): Service price: {price_text(r['Interested Service'])} | "
            f"Budget: ${int(r['Budget (USD)']):,} ({r['Budget Type']}) | Budget check: {'Yes' if f.budget_ok else 'No'} | "
            f"Decision maker: {'Yes' if f.dm else 'No'} | Qualified: {'Yes' if f.qualified else 'No'} | "
            f"Priority: {f.priority} | Days since last contact: {f.days} | Next action: {f.action} | Cross-sell: {cross} | "
            f"Cheaper options in the same service: {cheaper_options(r['Interested Service'])}")

print(df["Priority"].value_counts().to_dict(), "| at risk:", int(df["At Risk"].sum()))   # quick check of the rules
print(lead_facts(0))                                                            # example: Nile Pharma

## 4. Build the Search (Chunks -> Embeddings -> FAISS)
- **PDF:** cut into chunks (500 characters, 100 overlap). Each chunk remembers its **page**, so answers can cite it.
- **Sheet:** each row becomes one text line, stored in a second index (`leads_store`). If the question names a company, we take its row directly.

In [ ]:
lead_texts = [" | ".join(f"{col}: {row[col]}" for col in df.columns if col not in ("Priority", "At Risk"))   # one text line per lead
              for _, row in df.iterrows()]
print(lead_texts[0])                                                                # show the first lead as an example

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)        # chunks of 500 characters, 100 shared between neighbours
pdf_chunks = splitter.create_documents(pdf_pages, metadatas=[{"page": n + 1} for n in range(len(pdf_pages))])   # every chunk keeps its page
print(len(pdf_chunks), "PDF chunks")

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")             # small, fast embedding model: text -> vector

class VectorStore:                                                                    # a tiny FAISS index (cosine similarity)
    def __init__(self, texts, pages=None):
        self.texts, self.pages = list(texts), pages or [None] * len(texts)
        vectors = embedder.encode(self.texts, normalize_embeddings=True)              # unit vectors -> inner product = cosine
        self.index = faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(np.asarray(vectors, dtype="float32"))

    def search(self, query, k=4):                                                     # the k closest texts, with their page
        q = embedder.encode([query], normalize_embeddings=True).astype("float32")
        _, ids = self.index.search(q, k)
        return [(self.texts[i], self.pages[i]) for i in ids[0] if i >= 0]

pdf_store = VectorStore([c.page_content for c in pdf_chunks], [c.metadata["page"] for c in pdf_chunks])   # index 1: playbook
leads_store = VectorStore(lead_texts)                                                                     # index 2: leads
print(len(pdf_store.texts), "PDF chunks |", len(leads_store.texts), "leads stored")

for text, page in pdf_store.search("How much does the Business Website cost?", k=2):   # quick test
    print(f"- (page {page})", text[:110].replace("\n", " "))

## 5. Load the Model (Qwen 2.5 7B Instruct, 4-bit)

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"                                                   # the LLM we use (7B, instruction-tuned)
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)   # 4-bit -> fits in the free Colab T4

tokenizer = AutoTokenizer.from_pretrained(model_name)                                     # tokenizer: text <-> tokens
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto",
                                             quantization_config=bnb_config, dtype=torch.float16)   # float16: the T4 has no fast bfloat16

In [ ]:
def generate(messages, max_new_tokens=300):                                                     # send messages to Qwen, get the answer
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)   # messages -> one prompt in Qwen's chat format
    inputs = tokenizer(text, return_tensors="pt").to(model.device)                               # prompt -> tokens on the GPU
    with torch.no_grad():                                                                        # no training, only answering
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,        # greedy: same question, same answer
                                temperature=None, top_p=None, top_k=None,                         # unset sampling settings (no warnings)
                                pad_token_id=tokenizer.eos_token_id)
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]                                       # keep only the new answer
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()                        # tokens -> clean text

## 6. Finding the Lead(s)
- Longest names are checked first, so "Masr Digital Auto Center" is not mistaken for a shorter name inside it.
- Case and accents do not matter (*cafe masr* = *Café Masr*), a Lead ID like `L001` works, and a small typo (*Nile Pharam*) is still found.
- A question can name **several** companies.

In [ ]:
def normalize(text):                                                   # "Café Masr!" -> " cafe masr "
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(c for c in text if not unicodedata.combining(c)).lower()
    return " " + " ".join(re.findall(r"[a-z0-9]+", text)) + " "

NAMES = [normalize(n) for n in df["Company"]]                          # normalized company names
NAME_ORDER = sorted(range(len(NAMES)), key=lambda i: -len(NAMES[i]))   # longest names first
IDS = {lid.lower(): i for i, lid in enumerate(df["Lead ID"])}           # "l001" -> row number

def find_companies(question):                                          # all leads named in the question, in order
    text, found = normalize(question), {}
    for m in re.finditer(r"\bl\d{3}\b", text):                         # Lead IDs (L001)
        if m.group() in IDS: found.setdefault(IDS[m.group()], m.start())
    for i in NAME_ORDER:                                               # exact names
        pos = text.find(NAMES[i])
        if pos >= 0:
            found.setdefault(i, pos)
            text = text[:pos] + " " * len(NAMES[i]) + text[pos + len(NAMES[i]):]   # blank it so a shorter name inside is not matched
    if not found:                                                      # small typo: one clear fuzzy match only
        words, best, best_score = normalize(question).split(), None, 0
        for i, name in enumerate(NAMES):
            name = name.strip(); size = len(name.split())
            if len(name) < 8: continue                                 # short names match by accident too easily
            for s in range(max(1, len(words) - size + 1)):
                score = difflib.SequenceMatcher(None, name, " ".join(words[s:s + size])).ratio()
                if score > best_score: best, best_score = i, score
        if best_score >= 0.9: found[best] = 0
    return sorted(found, key=found.get)[:3]                            # at most 3 leads per question

def find_company(question):                                            # the first lead named (kept for the old tests)
    leads = find_companies(question)
    return leads[0] if leads else None

for q in ["Tell me about NILE PHARMA", "what about cafe masr?", "Brief me on L001", "Tell me about Nile Pharam",
          "Compare Nile Pharma and Sinai Tours", "What are the payment terms?"]:
    print(f"{q!r:45} ->", [df['Company'][i] for i in find_companies(q)])

## 7. Counting Questions With pandas
"How many leads are in Negotiation?" cannot be answered by search (search returns only a few rows).
So we detect these questions, filter the table with pandas, and give the model the exact numbers.

In [ ]:
PIPELINE_Q = re.compile(r"\b(how many|number of|count|which (?:\w+ ){0,2}(?:leads?|compan(?:y|ies)|clients?)|list|show me|"
                        r"pipeline|at[- ]risk|call first|prioriti[sz]e|top \d+|biggest budgets?)\b", re.I)
LEAD_NOUN = re.compile(r"\b(leads?|deals?|compan(?:y|ies)|clients?|pipeline|prioriti[sz]e|call first|at[- ]risk)\b", re.I)

def is_pipeline_question(question):                                     # about many leads, not one
    return bool(PIPELINE_Q.search(question) and LEAD_NOUN.search(question))

def pipeline_facts(question):                                           # filter the table and count, with pandas
    q, rows, used = question.lower(), df, []
    for col in ["Priority", "Stage", "Industry", "City", "Interested Service", "Lead Source"]:
        values = [v for v in df[col].dropna().unique() if re.search(r"\b" + re.escape(str(v).lower()) + r"\b", q)]
        values = [v for v in values if not any(v != w and str(v).lower() in str(w).lower() for w in values)]   # "Website" vs "Business Website"
        if values:
            rows = rows[rows[col].isin(values)]
            used.append(f"{col} = {' or '.join(map(str, values))}")
    if re.search(r"at[- ]risk", q):
        rows = rows[rows["At Risk"]]; used.append("At risk = yes")
    rows = rows.sort_values("Budget (USD)", ascending=False)               # biggest budget first
    days = facts["days"]                                                    # days since last contact, per lead
    top = ", ".join(f"{r.Company} ({r.Stage}, {r.Priority}, ${int(r['Budget (USD)']):,}, {days[i]} days since contact)"
                    for i, r in rows.head(10).iterrows())
    return (f"PIPELINE FACTS (computed with pandas from all {len(df)} leads, trust these): "
            f"Filter: {'; '.join(used) or 'all leads'} | Matching leads: {len(rows)} (say this number first) | "
            f"At risk means: an active lead with no contact for more than {AT_RISK_DAYS} days | "
            f"Top by budget: {top or 'none'}{' (and ' + str(len(rows) - 10) + ' more)' if len(rows) > 10 else ''} | "
            f"All leads by priority: {df['Priority'].value_counts().to_dict()} | All leads by stage: {df['Stage'].value_counts().to_dict()}\n\n")

print(pipeline_facts("How many leads are in Negotiation?")[:300])
print(pipeline_facts("Which hot leads are in Healthcare?")[:300])

## 8. Price Check (catches made-up prices)
If the answer writes a price right after a package name, the price must be that package's price
(or a discount, a 50/30/20 payment part, or a few months of a monthly price).
If not, the model is asked once more, and if the price is still wrong, a warning is added to the answer.

In [ ]:
PLAYBOOK_TEXT = "\n".join(pdf_pages)
PLAYBOOK_PERCENTS = {float(p) for p in re.findall(r"(\d+(?:\.\d+)?)\s?%", PLAYBOOK_TEXT)}   # every % in the playbook
MONEY = re.compile(r"\$\s?(\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?")
PACKAGE_RE = re.compile(r"\b(" + "|".join(re.escape(p) for p in sorted(PRICES, key=len, reverse=True)) + r")\b", re.I)

def allowed_prices(package):                                            # every correct way to write this package's price
    base = {PRICES[package]}
    if package in MONTHLY: base |= {PRICES[package] * m for m in (3, 6, 12)}           # 3, 6 or 12 months
    if package == "AI Chatbot": base |= {300, 5300, 5000 + 300 * 12}                  # setup + $300 / month
    out = set(base)
    for b in base:
        for p in PLAYBOOK_PERCENTS:
            if p < 100:
                out |= {b * p / 100, b * (100 - p) / 100}                              # a discount or the price after it
                out |= {b * (100 - p) / 100 * s / 100 for s in (50, 30, 20)}           # payment parts after a discount
    return out

def wrong_prices(answer):                                               # ["$5,000 for the Basic App", ...]
    problems, mentions = [], list(PACKAGE_RE.finditer(answer))
    for n, m in enumerate(mentions):
        end = mentions[n + 1].start() if n + 1 < len(mentions) else len(answer)
        window = answer[m.end():min(end, m.end() + 50)]                   # the text right after the package name
        money = MONEY.search(window)
        if not money or money.start() > 30 or any(w in window[:money.start()].lower() for w in ("budget", ".", ";", " and ", " or ")):
            continue                                                      # no price right after the name
        value = float(money.group(1).replace(",", ""))
        package = next(p for p in PRICES if p.lower() == m.group(1).lower())
        if not any(abs(value - a) <= 1 for a in allowed_prices(package)):
            problems.append(f"{money.group(0)} for the {package}")
    return problems

print(wrong_prices("The Basic App costs $5,000."))                          # caught: $5,000 is the AI Chatbot price
print(wrong_prices("With 10% off, the Advanced App is $22,500."))           # correct -> []

## 9. Chat Function (RAG + Memory)
1. **Find the lead(s):** companies named in the question; for *they / it / their*, the leads of the last turn.
2. **Get the context:** lead rows + computed facts + playbook rules. Counting questions get **pandas facts**. Otherwise search the playbook (FAISS + keywords) and, for lead questions, the leads index.
3. **Memory:** the last 5 questions and answers are sent with the new question.
4. **Check:** wrong package prices are fixed or flagged.

In [ ]:
# system prompt: the rules the model must follow in every answer
SYSTEM_PROMPT = """You are a Sales Assistant for Nova Digital Solutions (websites, mobile apps, digital marketing, AI solutions).
The user is a sales rep. Rules:
1. Answer ONLY from the Reference data and the chat history.
2. Use the chat history to understand "it", "they", "their".
3. "My name" means the USER's name. Take it ONLY from what the user said in the chat history. If the user never told you, say exactly: "I don't have this information."
4. Keep answers short (2 to 4 sentences). Never invent prices or features.
5. Answer ONLY what was asked. Give a BRIEF only if the user asks for a brief, a summary, or "tell me about the lead". For a simple question (stage, contact, price) answer in one sentence.
6. For a yes/no question, start with "Yes" or "No". Do not start with "Yes" or "No" for other questions.
7. Budget check, Priority, Qualified, Next action and Days since last contact must be copied from the COMPUTED FACTS line. Never recalculate them.
8. For a BRIEF, use one short line and ALWAYS include all of these: Company | Contact (job title) | Service and price | Budget check | Next action | Main problem (and any discount request in the notes) | Cross-sell idea (from COMPUTED FACTS) | Priority
9. Only say a package includes a feature if the Reference data says so. Do not mix packages (a Basic App is not a Basic Website).
10. Discounts: use only the discount that matches the lead (non-profit/education 20%, 2+ services 15%, 12-month marketing 10%). Non-profit only if the Industry says Non-profit or Education.
11. Price objection ("the price is too high"): follow the OBJECTION FACTS line. Never start with "Yes" and never agree that the price is too high. Suggest the first item in "Cheaper options in the same service" (never a package from another service), then the 15% bundle discount if they need 2+ services, and copy the payment split exactly as written in OBJECTION FACTS. Say the price is above the budget ONLY if Budget check is No.
12. For questions about many leads (how many, which leads, lists), use ONLY the PIPELINE FACTS. Start with the exact number of Matching leads, then list the names given. Never count rows yourself and never invent a reason why a lead matches.
13. When two leads are compared, use the COMPUTED FACTS line of each lead.
14. If the answer is not in the Reference data, say exactly: "I don't have this information." """

OBJECTION = re.compile(r"too (?:high|expensive)|expensive|can'?t afford|cheaper|lower (?:the )?price", re.I)
BRIEF = re.compile(r"\b(brief|summary|summari[sz]e|tell me about)\b", re.I)
YES_NO_START = ("is", "are", "can", "could", "does", "do", "did", "will", "would", "should", "has", "have", "was", "were")

def objection_facts(i):                                       # playbook sections 10 and 12, written for this lead
    f = facts.iloc[i]
    budget = ("Their budget covers the price, so do not say the price is too high for them; explain the value instead."
              if f.budget_ok else "Their budget is below the price.")
    return (f"OBJECTION FACTS for {df['Company'][i]} (from the playbook, trust these): Budget check: {'Yes' if f.budget_ok else 'No'}. {budget} "
            f"Suggest first: {cheaper_options(df['Interested Service'][i]).split(', ')[0]}. "
            f"Bundle discount: 15% if they buy 2 or more services. "
            f"Payment split for projects: 50% when signing, 30% at the middle of the project, and 20% at delivery.")

def clean_answer(question, answer, leads):                    # small fixes that do not need the model
    first = question.strip().split()[0].lower() if question.strip() else ""
    trimmed = re.sub(r"^(?:yes[,.!]?\s+)?(?:the\s+)?price is (?:indeed\s+)?too high[.!]\s*", "", answer, flags=re.I)
    answer = trimmed or answer                                # never open by agreeing that the price is too high
    if first not in YES_NO_START:                             # not a yes/no question -> no "Yes," / "No," at the start
        answer = re.sub(r"^(yes|no)[,.!]?\s+", "", answer, flags=re.I)
        answer = answer[:1].upper() + answer[1:]
    if BRIEF.search(question):                                # a brief must always show the computed priority
        for i in leads:
            if facts.iloc[i].priority.lower() not in answer.lower():
                answer += f"\nPriority ({df['Company'][i]}): {facts.iloc[i].priority}"
    return answer

RETRY = "These prices are wrong: {}. Answer the same question again and copy every price exactly from the Reference data."

In [ ]:
history = []                                                 # memory: list of past turns (question, answer, leads, sources)

STOP = set("the a an of to in for and or is are do does how what which who much many can you give me about my on with it they their them its this that".split())   # common words ignored in keyword search
LEAD_WORDS = {"lead", "leads", "company", "companies", "client", "clients", "contact", "budget", "stage"}   # words that mean the question is about leads
PRONOUNS = {"it", "they", "their", "them", "its", "he", "she", "his", "her", "both", "one"}                # words that point to the previous topic

def names_unknown_company(question):                         # detect a company that is NOT in our sheet
    """'lead Orion Mining' / 'about Orion Mining' with a Capitalised name that is not in the sheet and is not a package."""
    for m in re.finditer(r"\b(?:lead|company|client|about)\s+((?:[A-Z][\w&.'-]*)(?:\s+[A-Z][\w&.'-]*)*)", question):
        name = m.group(1).lower()
        if any(name in p.lower() or p.lower() in name for p in list(PRICES) + ["Nova Digital Solutions", "Nova Digital", "Playbook"]):
            continue                                         # a package or our own company, not a lead
        return m.group(1)
    return None

def keyword_docs(text, n=2):                                 # simple keyword search (backup for FAISS)
    q = {w for w in re.findall(r"[a-z0-9%+]+", text.lower()) if w not in STOP and len(w) > 2}
    scored = sorted(((len(q & set(re.findall(r"[a-z0-9%+]+", c.page_content.lower()))), c.page_content, c.metadata["page"])
                     for c in pdf_chunks), key=lambda x: -x[0])
    return [(t, p) for s, t, p in scored[:n] if s >= 2]      # top n chunks that share at least 2 words

def find_docs(text, k=4):                                    # hybrid search on the playbook: FAISS + keywords
    docs = pdf_store.search(text, k=k)
    docs += [d for d in keyword_docs(text) if d not in docs]
    return docs

def chat(question, show_sources=False):                      # main function: question -> answer
    words = set(re.findall(r"[a-z']+", question.lower()))

    # 1. which companies? (named in the question, or the ones from the last turn for it/they/their)
    leads = find_companies(question)
    if not leads:
        if names_unknown_company(question):                  # a company that is not in the sheet -> do not guess
            answer = "I don't have this information."
            history.append({"user": question, "assistant": answer, "leads": [], "sources": []})
            return answer
        if history and (words & PRONOUNS) and "my" not in words:
            leads = history[-1]["leads"]                     # "they" = the leads of the last turn

    # 2. follow-up question: add the old question to the search
    text = history[-1]["user"] + " " + question if history and (words & PRONOUNS) else question

    # 3. information
    sources, info = [], ""
    if not ("name" in words and "my" in words):              # "what is my name?" comes only from the chat history
        if not leads and is_pipeline_question(question):     # counting / list question -> pandas
            info += pipeline_facts(question); sources.append("pandas: all leads")
        docs = find_docs(text)
        for lead in leads:                                   # each lead: row + computed facts + rules for its service and stage
            info += lead_texts[lead] + "\n" + lead_facts(lead) + "\n"
            if OBJECTION.search(question): info += objection_facts(lead) + "\n"   # price objection -> exact playbook answer
            info += "\n"
            docs += [d for d in find_docs(df["Interested Service"][lead] + " " + df["Stage"][lead]) if d not in docs]
            sources.append(f"CRM row {df['Lead ID'][lead]} ({df['Company'][lead]})")
        if not leads and (words & LEAD_WORDS) and not is_pipeline_question(question):
            info += "\n".join(t for t, _ in leads_store.search(text, k=3)) + "\n\n"   # the 3 closest leads
            sources.append("CRM search")
        info += "\n\n".join(t for t, _ in docs) + "\n\n"
        sources += sorted({f"Playbook page {p}" for _, p in docs})

    # 4. messages: rules + last 5 turns + new question with its data
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history[-5:]:
        messages += [{"role": "user", "content": turn["user"]}, {"role": "assistant", "content": turn["assistant"]}]
    messages.append({"role": "user", "content": "Reference data:\n" + info + "Question: " + question})

    # 5. answer, price check, save
    answer = generate(messages)
    problems = wrong_prices(answer)
    if problems:                                             # ask once more, telling the model what was wrong
        second = generate(messages + [{"role": "assistant", "content": answer},
                                      {"role": "user", "content": RETRY.format(", ".join(problems))}])
        if len(wrong_prices(second)) < len(problems): answer = second
        if wrong_prices(answer): answer += "\n(Check: " + ", ".join(wrong_prices(answer)) + " is not the playbook price.)"
    answer = clean_answer(question, answer, leads)
    if answer.startswith("I don't have this information"): sources = []   # nothing was used
    history.append({"user": question, "assistant": answer, "leads": leads, "sources": sources})
    return answer + (f"\n   [sources: {'; '.join(sources) or 'chat history'}]" if show_sources else "")

## 10. Demo

In [ ]:
history.clear()                                                         # start with empty memory
for q in ["How much does the Business Website cost?",                   # PDF question (price)
          "Give me a short brief about Nile Pharma before my call.",    # lead brief (Sheet + rules)
          "What service are they interested in?",                       # "they" = Nile Pharma (memory)
          "They said the price is too high. How should I respond?",     # objection handling (memory + rules)
          "Compare Nile Pharma and Sinai Tours. Which one has the bigger budget?",   # two companies
          "How many leads are in Negotiation?",                         # counting with pandas
          "Which leads are at risk in Healthcare?"]:                    # at risk + filter with pandas
    start = time.time()
    print("Rep:", q)
    print("Bot:", chat(q, show_sources=True), f"({time.time() - start:.1f} s)")
    print("-" * 40)

history.clear()                                                         # a new chat
print("Rep: Who won the World Cup in 2018?")                            # out-of-scope question
print("Bot:", chat("Who won the World Cup in 2018?"))                   # not in our data -> "I don't have this information"

## 11. Evaluation
Each test is a question and a word that must appear in the answer (and optionally a word that must NOT appear).
Tests with a `first` message check the memory.

- **Dev tests** are the questions used while writing the prompt (the original 30).
- **New tests** were written after the prompt was finished, for the new features and as a fairer check (the limitation from the first version).
- The expected numbers for counting tests are computed with pandas, not typed by hand.

In [ ]:
n_negotiation = int((df["Stage"] == "Negotiation").sum())             # expected answers computed from the data
n_hot = int((df["Priority"] == "Hot").sum())
n_healthcare = int((df["Industry"] == "Healthcare").sum())

# (set, type, first message(s) or None, question, expected word [, word that must NOT appear])
tests = [
    ("dev", "PDF", None, "How much does the Business Website cost?", "4000"),
    ("dev", "PDF", None, "How much does the Basic Website cost?", "1500"),
    ("dev", "PDF", None, "How much does the Basic App cost?", "6000"),
    ("dev", "PDF", None, "What is the price of the Advanced App?", "25000"),
    ("dev", "PDF", None, "How many weeks does it take to deliver the Advanced App?", "16 weeks"),
    ("dev", "PDF", None, "What discount do education organizations get?", "20%"),
    ("dev", "PDF", None, "What is the maximum discount a sales rep can give without approval?", "10%"),
    ("dev", "PDF", None, "What is the commission on AI Solutions?", "7%"),
    ("dev", "Sheet", None, "What industry is Nile Pharma in?", "healthcare"),
    ("dev", "Sheet", None, "Who is the contact person at Delta Foods?", "karim"),
    ("dev", "Sheet", None, "What stage is Cairo Logistics in?", "negotiation"),
    ("dev", "Sheet", None, "What is the budget of Royal Hotels?", "30000"),
    ("dev", "Reasoning", None, "Can Royal Hotels afford the Advanced App?", "yes"),
    ("dev", "Memory", "Tell me about the lead Nile Pharma.", "How many employees do they have?", "85"),
    ("dev", "Memory", "Tell me about the lead Delta Foods.", "What stage are they in?", "contacted"),
    ("dev", "Memory", "What is the Business Website package?", "How many weeks does it take?", "6 weeks"),
    ("dev", "Memory", "What is the Standard App package?", "How much does it cost?", "12000"),
    ("dev", "Memory", "Hi, my name is Hagar.", "What is my name?", "hagar"),
    ("dev", "PDF", None, "What is the commission on website and app projects?", "5%"),
    ("dev", "Reasoning", None, "Give me a short brief about Royal Hotels.", "warm"),
    ("dev", "Reasoning", None, "Give me a short brief about MedLine Labs.", "warm"),
    ("dev", "Reasoning", None, "Give me a short brief about Nile Pharma.", "hot"),
    ("dev", "Reasoning", None, "Give me a short brief about Desert Farms.", "cold"),
    ("dev", "Reasoning", None, "What discount can Elite Clinics get for a 12-month contract?", "10%", "non-profit"),
    ("dev", "Memory", ["Tell me about the lead Cairo Logistics."], "Who is their contact person?", "hany saeed"),
    ("dev", "Out of scope", ["Tell me about the lead Elite Clinics."], "Tell me about the lead Orion Mining.", "don't have this information"),
    ("dev", "Out of scope", ["Tell me about the lead Elite Clinics."], "What is my name?", "don't have this information"),
    ("dev", "Reasoning", ["Give me a short brief about Nile Pharma before my call."], "They said the price is too high. How should I respond?", "standard app", "basic website"),
    ("dev", "Out of scope", None, "Who won the World Cup in 2018?", "don't have this information"),
    ("dev", "Out of scope", None, "Tell me about the lead Orion Mining.", "don't have this information"),
    # ---- new tests (written after the prompt was final) ----
    ("new", "Counting", None, "How many leads are in Negotiation?", str(n_negotiation)),
    ("new", "Counting", None, "How many hot leads do we have?", str(n_hot)),
    ("new", "Counting", None, "How many leads are in the Healthcare industry?", str(n_healthcare)),
    ("new", "Two leads", None, "Compare Nile Pharma and Sinai Tours. Which one has the bigger budget?", "nile pharma"),
    ("new", "Two leads", None, "What do Nile Pharma and Delta Foods want?", "advanced app"),
    ("new", "Name match", None, "What stage is cairo logistics in?", "negotiation"),
    ("new", "Name match", None, "What industry is Nile Pharam in?", "healthcare"),
    ("new", "Name match", None, "Who is the contact person for L001?", "mona"),
    ("new", "Reasoning", None, "Is the budget of Sinai Tours enough for the service they want?", "no"),
    ("new", "Reasoning", None, "Is the contact person at Desert Farms a decision maker?", "no"),
    ("new", "PDF", None, "What are the payment terms for projects?", "50%"),
    ("new", "PDF", None, "How long is the minimum contract for SEO?", "6 months"),
    ("new", "Memory", "Tell me about the lead Sinai Tours.", "What is their budget?", "4000"),
    ("new", "Out of scope", None, "What is the weather in Cairo today?", "don't have this information"),
    ("new", "Reasoning", ["Give me a short brief about Nile Pharma before my call."], "They said the price is too high. How should I respond?", "middle", "after 30 days"),
    ("new", "Counting", None, "How many leads are at risk in Healthcare?", str(int(((df["Industry"] == "Healthcare") & df["At Risk"]).sum()))),
]

def run_test(first, question, expected, bad=None):                    # run one test
    history.clear()                                                    # every test starts with empty memory
    for f in ([first] if isinstance(first, str) else (first or [])):   # first message(s), if any
        chat(f)
    start = time.time()
    answer = chat(question)
    clean = answer.lower().replace(",", "").replace("\u2019", "'")     # "4,000" -> "4000", fix the apostrophe
    return (expected.lower() in clean) and not (bad and bad in clean), answer, time.time() - start

rows = []
for test_set, kind, first, q, expected, *bad in tests:
    ok, answer, seconds = run_test(first, q, expected, *bad)
    rows.append({"set": test_set, "type": kind, "question": q, "correct": ok, "seconds": round(seconds, 1), "answer": answer})
results = pd.DataFrame(rows)

for name, part in results.groupby("set"):
    print(f"{name} tests: {part['correct'].sum()} / {len(part)} correct ({part['correct'].mean():.0%})")
print("Average time per answer:", round(results["seconds"].mean(), 1), "s")
print(results.groupby(["set", "type"])["correct"].mean().mul(100).round())

In [ ]:
wrong = results[~results["correct"]]                  # keep only the failed tests
for _, r in wrong.iterrows():
    print("X", f"[{r['set']}]", r["question"], "\n  ->", r["answer"][:200], "\n")
print("All tests passed!" if wrong.empty else f"{len(wrong)} test(s) failed")

## 12. Chat With the Assistant
Type `quit` to stop, `clear` to start a new chat, `sources` to show or hide the sources.

In [ ]:
history.clear()
show = False                                         # show the sources under each answer?
while True:
    question = input("You: ").strip()
    if question.lower() == "quit":
        break
    elif question.lower() == "clear":
        history.clear(); print("Memory cleared.")
    elif question.lower() == "sources":
        show = not show; print("Sources", "on" if show else "off")
    elif question:
        print("Assistant:", chat(question, show_sources=show), "\n")

## 13. Conclusion
**What we built:** an assistant that reads a PDF playbook and a Google Sheet of 200 leads, finds the right information
(FAISS + keyword search + company-name matching), and answers with Qwen 2.5 7B. It remembers the last 5 messages.

**What improved in this version**
- Counting and list questions are answered with **pandas**, so the numbers are exact.
- **Two companies** in one question, and names with different case, accents, Lead IDs or small typos.
- **At-risk leads** and the response time by lead source, from the playbook rules.
- A **price check** that catches a wrong price for a package and asks the model again.
- **Sources** (CRM row, playbook page) for every answer.
- **New test questions** written after the prompt, so the accuracy is not only measured on the questions used to tune it.

**Limitations**
- The 7B model can still mix up words or features; the price check only covers prices written next to a package name.
- The pandas filters understand the column values (stage, priority, industry, city, service, source), not free-form conditions like "budget above 10,000".
- The test set is still small (46 questions) and checks for one keyword per answer.

**Future work:** an LLM-as-judge evaluation on a bigger test set, a feature check like the price check, and a real CRM connection instead of a sheet.
This notebook is also the base of the full app (FastAPI + web UI + 89 automated tests): https://github.com/Hagar202/nova-sales-assistant